# Heritage Short: ACE-Step 1.5 score candidates (Kaggle or Colab)

One instrumental candidate per seed in `music.seeds`, from the caption, bpm and key in `storyboard.json`.
**Not run on the author's machine (no GPU); first run on Kaggle.** API names follow
`docs/en/INFERENCE.md` in the ACE-Step 1.5 repo; if a name has moved, check that file.

1. Attach the same `heritage-job` dataset (from `make kaggle-export`). GPU T4, Internet on.
2. Run all, download `heritage_results.zip`, then `make import ZIP=...`.
3. Listen to each `outputs/music/candidates/seed_*.wav` **under the narration** (`make radio`),
   set `music.chosen_seed` and `music.choice_reason` in `storyboard.json`, and log the comparison in EXPERIMENTS.md.

In [ ]:
!git clone -q https://github.com/ACE-Step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5 || true
!pip install -q -e /kaggle/working/ACE-Step-1.5

In [ ]:
import glob, json, os, socket, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

JOB = Path(glob.glob("/kaggle/input/**/jobs.json", recursive=True)[0]).parent \
    if glob.glob("/kaggle/input/**/jobs.json", recursive=True) else Path("job")
jobs = json.loads((JOB / "jobs.json").read_text())
OUT = Path("/kaggle/working/results") if Path("/kaggle/working").exists() else Path("results")
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "runs_kaggle.jsonl"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-kaggle"

def gpu_name():
    try:
        import torch
        return torch.cuda.get_device_name(0)
    except Exception:
        return "unknown"

def log_call(stage, shot, model, params, seed, latency_s, status, output=None, error=None):
    """Same shape as pipeline/common.py log lines, so make import can merge them."""
    rec = {"ts": datetime.now(timezone.utc).isoformat(timespec="milliseconds"), "run_id": RUN_ID,
           "event": "model_call", "stage": stage, "shot": shot, "provider": "kaggle", "model": model,
           "params": params, "seed": seed, "output": output, "attempt": 1, "status": status,
           "latency_s": round(latency_s, 2), "gpu": gpu_name(),
           "cost": {"gpu_minutes": round(latency_s / 60, 2), "tier": "free (Kaggle weekly GPU quota)"}}
    if error:
        rec["error"] = str(error)[:300]
    with LOG.open("a") as f:
        f.write(json.dumps(rec) + "\n")
    print(f"{status:5} {shot or ''} {latency_s:7.1f}s {output or error or ''}")

print("job folder:", JOB, "| storyboard version", jobs["storyboard_version"])

In [ ]:
import shutil, sys
REPO = "/kaggle/working/ACE-Step-1.5" if Path("/kaggle/working").exists() else "ACE-Step-1.5"
sys.path.insert(0, REPO)
from acestep.handler import AceStepHandler
from acestep.inference import GenerationConfig, GenerationParams, generate_music

DIT = "acestep-v15-turbo"
dit = AceStepHandler()
dit.initialize_service(project_root=REPO, config_path=DIT, device="cuda")

# The optional LM planner refines the caption. Try the small one; without it, generation still works.
llm = None
try:
    from acestep.llm_inference import LLMHandler
    llm = LLMHandler()
    llm.initialize(checkpoint_dir=f"{REPO}/checkpoints", lm_model_path="acestep-5Hz-lm-0.6B",
                   backend="pt", device="cuda")
except Exception as err:
    print("LM planner not loaded, continuing without it:", err)
    llm = None

In [ ]:
M = jobs["music"]
(OUT / "music").mkdir(exist_ok=True)
for seed in M["seeds"]:
    params = {"caption": M["caption"], "bpm": M["bpm"], "keyscale": M["keyscale"], "duration": M["duration"],
              "instrumental": True, "inference_steps": 8, "thinking": llm is not None, "dit": DIT}
    t0 = time.perf_counter()
    try:
        result = generate_music(
            dit, llm,
            GenerationParams(caption=M["caption"], lyrics="[Instrumental]", instrumental=True,
                             duration=M["duration"], bpm=M["bpm"], keyscale=M["keyscale"], seed=seed,
                             inference_steps=8, thinking=llm is not None),
            GenerationConfig(batch_size=1, audio_format="wav"),
            save_dir=str(OUT / "tmp"))
        if not result.success:
            raise RuntimeError(result.error)
        src = Path(result.audios[0]["path"])
        dst = OUT / "music" / f"seed_{seed}{src.suffix}"
        shutil.copy(src, dst)
        log_call("music", None, "ACE-Step 1.5 " + DIT, params, seed, time.perf_counter() - t0, "ok",
                 output=f"outputs/music/candidates/{dst.name}")
    except Exception as err:
        log_call("music", None, "ACE-Step 1.5 " + DIT, params, seed, time.perf_counter() - t0, "error", error=err)
shutil.rmtree(OUT / "tmp", ignore_errors=True)

In [ ]:
# Zip everything in results/ so one download brings it home: then `make import ZIP=...`
zpath = OUT.parent / "heritage_results.zip"
with zipfile.ZipFile(zpath, "w") as z:
    for f in OUT.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(OUT.parent))
print("download:", zpath, round(zpath.stat().st_size / 1e6, 1), "MB")